# Lab 1 - The Reliable Extractor (Track N: notebook)

One file, top to bottom. Work the cells in order, run each one, read its
output. Nothing here asks you to open another script.

This repo has two tracks. **This notebook is Track N.** The alternative is
Track S - the scripts `v0_naive.py`, `extract.py`, `run_eval.py` under
`labs/lab1/`. Pick one and ignore the other for the whole lab.

| Part | What you do | ~time |
|---|---|---|
| 0 | Environment + data check | 5 min |
| A | Watch the naive version fail | 20 min |
| B | Schema, validation, repair loop | 45 min |
| C | Move deterministic fields out of the model | 30 min |
| D | Measure the triple and analyse | 35 min |
| Submit | Copy your code into `extract.py` + fill `report.md` | 5 min |

**The one rule:** run cells in order, and when a cell says *your turn*, edit it
there. When you are done, copy the schema and functions you wrote into
`labs/lab1/extract.py` - that is the graded file.

## 0 - Setup

In [3]:
# Make the shared library importable and load settings. Safe to re-run.
import os, sys, json, re
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "aip").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "aip").is_dir():
    ROOT = Path.cwd().parents[1]          # notebook opened from labs/lab1
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.chdir(ROOT)
print("repo root:", ROOT)
print("python:", sys.version.split()[0])

repo root: c:\Users\chou_\OneDrive - Plaksha University\Documents\msai_projects\msai\practice_lab\AI-in-Practice-Lab
python: 3.12.10


In [ ]:
pip install -r requirements.txt

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached jupyterlab-4.6.4-py3-none-any.whl.metadata (16 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.5-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jupyter_builder-1.2.3-py3-none-any.whl.metadata (7.8 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.21.1-py3-none-any.whl.meta


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
# Check dependencies and the API key - the notebook's check_setup.py.
ok = True
for mod in ("pydantic", "httpx", "dotenv"):
    try:
        __import__(mod)
        print("PASS  import " + mod)
    except ImportError:
        ok = False
        print("FAIL  import " + mod + "  ->  pip install -r requirements.txt")

have_key = bool(os.getenv("GEMINI_API_KEY", "").strip())
print(("PASS" if have_key else "FAIL") + "  GEMINI_API_KEY set")
if not have_key:
    print("      create .env in the repo root with GEMINI_API_KEY=AIza...")
print("Environment is ready." if (ok and have_key) else "Fix the FAILs above.")

PASS  import pydantic
PASS  import httpx
PASS  import dotenv
PASS  GEMINI_API_KEY set
Environment is ready.


In [6]:
# Load the golden sets. dev = iterate here. test = run once, at the end.
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

dev  = load_jsonl(ROOT / "data/eval/extraction_dev.jsonl")
test = load_jsonl(ROOT / "data/eval/extraction_test.jsonl")
print("dev:", len(dev), "tickets   test:", len(test), "tickets")

for r in dev[:5]:
    print("=" * 70)
    print("gold:", json.dumps(r["expected"]))
    print("-" * 70)
    print(r["input"][:600])

dev: 60 tickets   test: 120 tickets
gold: {"category": "complaint", "urgency": 4, "sentiment": "angry", "policy_number": null, "product": "unknown", "escalate": true, "contains_pii": false, "language": "en"}
----------------------------------------------------------------------
Your agent mis-sold me this policy. Nobody told me maternity has a 5-month waiting period when I bought my policy. I want a full refund.

Yours sincerely,
Tenzin Dsouza
gold: {"category": "policy_change", "urgency": 3, "sentiment": "frustrated", "policy_number": null, "product": "bronze", "escalate": false, "contains_pii": true, "language": "en"}
----------------------------------------------------------------------
[via WhatsApp Business]
I submitted a portability request 21 days ago and heard nothing. My existing policy renews soon. Reference is my Aurora Bronze policy.

Thanks & Regards,
Lakshmi Dsouza
Mob: 9591079365

> On 11 Mar 2026, Aurora Support <support@aurorahealth.example> wrote:
> Dear Customer, tha

## A - The naive version

The version everyone writes first: ask for JSON, then `json.loads` it. Run it
and watch how it fails. **Do not fix it here** - this is the exhibit.

In [7]:
from aip import chat, Budget

NAIVE_PROMPT = (
    "Extract the following from this support ticket and return JSON: "
    "category, urgency, sentiment, policy_number, product." + chr(10) + chr(10) +
    "Ticket:" + chr(10) + "{ticket}"
)

def extract_v0(ticket):
    # The naive version: trust the model, bare json.loads.
    return json.loads(chat(NAIVE_PROMPT.format(ticket=ticket),
                           tier="SMALL", max_tokens=300))

In [8]:
# Characterise the failures on 20 tickets. Fewer calls = faster = cheaper.
from collections import Counter

blocked = Counter()
parse_json_fail = 0

with Budget(limit_usd=0.05, label="part-a-naive") as b:
    for r in dev[:20]:
        try:
            extract_v0(r["input"])
        except json.JSONDecodeError:
            parse_json_fail += 1
        except Exception as exc:
            blocked[type(exc).__name__] += 1

print("bare json.loads() failed to parse :", parse_json_fail, "/20")
print("other exceptions                :", dict(blocked) or "none")

raw = chat(NAIVE_PROMPT.format(ticket=dev[0]["input"]), tier="SMALL", max_tokens=300)
print()
print("raw first 300 chars of one reply:")
print(repr(raw[:300]))

bare json.loads() failed to parse : 20 /20
other exceptions                : none

raw first 300 chars of one reply:
'```json\n{\n  "category": "Complaint / Mis-selling",\n  "urgency": "High",\n  "sentiment": "Negative",\n  "policy_number": null,\n  "product": "Maternity Insurance"\n}\n```'


**Write down the arc.** Bare parsing fails on almost every ticket, and the
reason is almost always the same wrapper - not bad JSON. That gap (it parses
once you strip the wrapper, but is still wrong inside) is why Part B exists.

Fill this in your report:

| Failure mode | Count in 20 |
|---|---|
| Not valid JSON at all | |
| Wrapped in a markdown fence | |
| Extra prose around the JSON | |
| Parses, but a field is wrong | |

## B - Schema, validation, repair loop

Define a Pydantic model (the contract), write a short system prompt, and call
`aip.llm.structured` - it parses tolerantly, validates, and repairs. You never
crash.

The field `description`s are sent to the model as part of the JSON Schema, so
write them as *instructions*, not documentation.

In [9]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator
from aip.llm import structured, StructuredOutputError

CATEGORIES = Literal["billing", "claims", "policy_change",
                     "technical", "complaint", "information"]

class TicketRecord(BaseModel):
    # YOUR TURN - fill in the descriptions and the missing fields.
    # Keep evidence BEFORE category: the model writes the quote first, then
    # classifies. Field order = schema order = attention order.

    evidence: str = Field(max_length=200, description="TODO B1")
    category: CATEGORIES = Field(description="TODO B1: define each of the six")
    urgency: int = Field(ge=1, le=5, description="TODO B1: anchor 1, 3, 5")
    sentiment: Literal["angry", "frustrated", "neutral", "satisfied"] = Field(
        description="TODO B1")
    product: Literal["bronze", "silver", "gold", "platinum", "unknown"] = Field(
        description="TODO B1")
    language: Literal["en", "hi-en"] = Field(description="TODO B1")

    policy_number: str | None = Field(default=None, description="TODO B1")
    contains_pii: bool = Field(default=False, description="TODO B1")

    needs_human_review: bool = False     # set by us, never the model
    review_reason: str = ""

    @field_validator("policy_number")
    @classmethod
    def _policy_format(cls, v):
        # TODO B1j: keep only exactly AUR-<7 digits>, else None (never raise).
        return v

SYSTEM_PROMPT = "TODO B2: write the system prompt (short, stable first)."

In [10]:
def _review(reason):
    return TicketRecord(evidence="", category="information", urgency=1,
                        sentiment="neutral", product="unknown", language="en",
                        needs_human_review=True, review_reason=reason)

def extract_b(ticket):
    # YOUR TURN - same TODOs as extract.py B3/B4:
    # TODO B3: call structured(...) with TicketRecord and SYSTEM_PROMPT.
    #          (max_tokens=4096 is a sane budget on the SMALL model.)
    # TODO B4: catch StructuredOutputError and return a record with
    #          needs_human_review=True. This function must never raise.
    raise NotImplementedError

In [11]:
# Quick smoke test on 3 tickets (cheap).
for r in dev[:3]:
    try:
        rec = extract_b(r["input"])
        print(r["id"], rec.category, rec.urgency, "review=" + str(rec.needs_human_review))
    except NotImplementedError:
        print("Complete the extract_b cell first.")
        break

Complete the extract_b cell first.


### Run the eval harness (built in)

`run_here` below is the notebook's version of `run_eval.py` - same metric, same
numbers.

In [12]:
from aip.evals import field_accuracy

GRADED_FIELDS = ["category", "urgency", "sentiment", "policy_number",
                 "product", "escalate", "contains_pii", "language"]

def make_metric(fields):
    def metric(pred, gold):
        if hasattr(pred, "model_dump"):
            pred = pred.model_dump()
        m = field_accuracy(pred, gold, fields)
        m["schema_valid"] = float(isinstance(pred, dict) and bool(pred))
        m["needs_review"] = float(bool(isinstance(pred, dict)
                                       and pred.get("needs_human_review")))
        return m
    return metric

def run_here(system, rows, fields, label, budget=0.30):
    from aip import Budget
    results = []
    with Budget(limit_usd=budget, label=label) as b:
        for i, r in enumerate(rows, 1):
            try:
                out = system(r["input"])
                results.append((r["id"], make_metric(fields)(out, r["expected"]), None))
            except Exception as exc:
                results.append((r["id"], {"error": 1.0},
                                type(exc).__name__ + ": " + str(exc)))
            print("  " + label + ": " + str(i) + "/" + str(len(rows)), end=chr(13))
    print(" " * 40, end=chr(13))
    return results, b.as_dict()

def summarise(results, b):
    n = len(results)
    def avg(key):
        vals = [m.get(key) for _, m, _ in results if key in m]
        return sum(vals) / len(vals) if vals else 0.0
    n_err = sum(1 for _, _metrics, err in results if err)

    record = sum(1 for _, m, err in results
                 if not err and m.get("record_accuracy", 0) == 1.0)
    print("  rows              ", n)
    print("  schema_valid      ", round(avg("schema_valid"), 3))
    print("  field_accuracy    ", round(avg("field_accuracy"), 3))
    print("  record_accuracy   ", round(record / n, 3))
    print("  errors            ", n_err)
    print("  cost              $", round(b.get("cost_usd", 0), 4))
    print("  latency p95       ", round(b.get("latency_p95_ms", 0)), "ms")

In [13]:
# YOUR TURN (run when extract_b is done): evaluate on dev.
B_FIELDS = [f for f in GRADED_FIELDS if f != "escalate"]
b_results, b_budget = run_here(extract_b, dev, B_FIELDS, "part-b", budget=0.30)
print("Variant B:")
summarise(b_results, b_budget)

Variant B:                              
  rows               60
  schema_valid       0.0
  field_accuracy     0.0
  record_accuracy    0.0
  errors             60
  cost              $ 0.0
  latency p95        0 ms


## C - Move deterministic work out of the model

Three fields should never be the model's problem: `policy_number` (regex),
`contains_pii` (fixed patterns), and `escalate` (a plain business rule).
Compute them in code - exact, auditable, free.

**The trap:** some tickets contain two policy-number-shaped strings, one in the
live body and one in a quoted reply (lines starting with `>`). Decide a rule and
state it in the comment.

In [14]:
POLICY_RE = re.compile(r"\bAUR-\d{7}\b")
QUOTED_LINE = re.compile(r"^\s*>.*$", re.MULTILINE)

def extract_deterministic(ticket):
    # YOUR TURN - same TODOs as extract.py C1/C3.
    # policy_number: Find AUR-<7 digits>. The trap: some tickets have TWO
    #   policy-shaped strings - one in the live body, one under '>' quoted
    #   lines. Decide a rule, write it in a comment, implement it.
    # contains_pii: True if the ticket has a phone number or an email that
    #   is NOT one of Aurora's own (support@ / grievance@aurorahealth.example).
    #   aip.guards.PII_PATTERNS has the patterns.
    raise NotImplementedError

def apply_business_rules(fields, ticket):
    # YOUR TURN - same TODO as extract.py C1b:
    #   escalate = urgency >= 4 or "ombudsman" appears in the ticket
    raise NotImplementedError

In [15]:
class TicketRecordC(BaseModel):
    # YOUR TURN - TODO C2: the reduced schema the model sees in Part C.
    # Copy TicketRecord and delete the fields you now compute in code
    # (policy_number, contains_pii). Fewer fields = shorter prompt,
    # fewer output tokens, those fields at 100% accuracy.
    pass

def _review_c(reason):
    return TicketRecordC(evidence="", category="information", urgency=1,
                         sentiment="neutral", product="unknown", language="en",
                         needs_human_review=True, review_reason=reason)

def extract_c(ticket):
    # YOUR TURN - model for judgement, code for everything else.
    # TODO B3/B4 pattern: structured(...) with TicketRecordC;
    #   catch StructuredOutputError -> _review_c(...);
    #   then out.update(extract_deterministic(ticket));
    #   return apply_business_rules(out, ticket).
    # Returns a plain dict. Must never raise.
    raise NotImplementedError

In [16]:
# Evaluate variant C on dev, next to B (run when extract_c is done).
c_results, c_budget = run_here(extract_c, dev, GRADED_FIELDS, "part-c", budget=0.30)
print("Variant C:")
summarise(c_results, c_budget)
print()
if "b_results" in dir():
    print("Variant B was:")
    summarise(b_results, b_budget)
    print()
    delta = b_budget.get("cost_usd", 0) - c_budget.get("cost_usd", 0)
    pct = (delta / b_budget["cost_usd"] * 100) if b_budget.get("cost_usd") else 0
    print("cost change: $" + format(delta, "+.4f") + "  (" + format(pct, "+.0f") + "%)")

Variant C:                              
  rows               60
  schema_valid       0.0
  field_accuracy     0.0
  record_accuracy    0.0
  errors             60
  cost              $ 0.0
  latency p95        0 ms

Variant B was:
  rows               60
  schema_valid       0.0
  field_accuracy     0.0
  record_accuracy    0.0
  errors             60
  cost              $ 0.0
  latency p95        0 ms

cost change: $+0.0000  (+0%)


## D - Measure and analyse

**The triple:** quality x cost x latency. Never quality alone.

In [17]:
# Per-field accuracy for C (worst first) - your error analysis.
def per_field(results):
    keys = [k for k in results[0][1] if k.startswith("field.")]
    agg = {}
    for k in keys:
        vals = [m[k] for _, m, _ in results if k in m]
        agg[k] = sum(vals) / len(vals) if vals else 0.0
    return dict(sorted(agg.items(), key=lambda kv: kv[1]))

for k, v in per_field(c_results).items():
    print("  " + format(k, "<28") + format(v, ".3f"))

In [18]:
# The worst records and which fields they got wrong.
bad = [(i, m) for i, m, err in c_results
       if not err and m.get("record_accuracy", 0) < 1]
print(len(bad), "imperfect records; showing up to 8:")
for i, m in bad[:8]:
    wrong = [k.removeprefix("field.") for k, v in m.items()
             if k.startswith("field.") and v == 0.0]
    print("  " + i + "  wrong=" + str(wrong))

0 imperfect records; showing up to 8:


In [19]:
# YOUR TURN: cluster the failures. Pick 15, group them, name the top three.
# Then the economic argument:
#   cost/ticket x 10,000/day   vs   40s of agent time at Rs300/hour
#   state the break-even accuracy and show the arithmetic.

# D1: run the test split ONCE when you are happy with C. Comment back in:
# test_results, test_budget = run_here(extract_c, test, GRADED_FIELDS, "part-d-test", budget=0.30)
# summarise(test_results, test_budget)

## Submit - turn the notebook into the graded files

You built everything here. When you are done:

1. Copy your schema and the `extract_*` functions into
   `labs/lab1/extract.py` (the only file graded).
2. Fill in `labs/lab1/report.md` yourself. Never regenerate it - the tables
   are yours to complete from the harness output.

### Cheat sheet

- `structured(prompt, schema=Model, system=..., tier="SMALL")` -> a validated
  instance, or raises `StructuredOutputError`. Tolerant parse plus a repair loop.
- `Budget(limit_usd=..., label=...)` as a context manager -> `.report()`,
  `.as_dict()`.
- Run the whole notebook top-to-bottom before submitting.
- Targets and checkpoints: `labs/lab1/README.md`.
- Procedure and commands: `labs/lab1/RUNSHEET.md`.
- Theory and traps: `labs/lab1/CONCEPTS.md`.